In [5]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

GOLD_PATH = Path('../../data/gold')
MODEL_DATA_PATH = Path('../../data/model_input')

MODEL_DATA_PATH.mkdir(parents=True, exist_ok=True)

print("Ambiente configurado.")
print(f"Lendo de: {GOLD_PATH.resolve()}")
print(f"Salvando em: {MODEL_DATA_PATH.resolve()}")

Ambiente configurado.
Lendo de: /home/pedroconrado/mortalidade-cardiovascular-ml/data/gold
Salvando em: /home/pedroconrado/mortalidade-cardiovascular-ml/data/model_input


## Carregar e Filtrar Colunas

vou remover algumas colunas que não poderia ter aqui.

In [9]:
# Carregamento do dataset Ouro
arquivo_gold = GOLD_PATH / 'dataset_modelagem.parquet'
try:
    df = pd.read_parquet(arquivo_gold)
    print(f'Dataset carregado com sucesso. Dimensoes: {df.shape}')
except Exception as e:
    print(f'Falha ao carregar dataset: {e}')

# Definicao da Variavel Target (Classe de Risco)
# Criterio: Taxa >= 150 mortes/100k habitantes eh considerado Alto Risco (1)
CUTOFF = 150
df['target'] = (df['taxa_mortalidade'] >= CUTOFF).astype(int)

print('\nDistribuicao da Classe Alvo:')
print(df['target'].value_counts(normalize=True))

#  Identificacao das 100 Maiores Cidades (para analise de recorte posterior)
# Seleciona as 100 maiores populacoes baseadas no ano mais recente (2015)
top_100_codes = df[df['ano'] == 2015].nlargest(100, 'populacao')['codmun'].unique()
df['is_top_100'] = df['codmun'].isin(top_100_codes).astype(int)

print(f'\nRegistros pertencentes as 100 maiores cidades: {df["is_top_100"].sum()}')

# 3. Selecao de Features (Remocao de vazamento de dados e identificadores)
# Colunas a remover: nomes, chaves de processamento e variaveis que compoem a resposta (target leakage)
cols_to_drop = [
    'nome_origem', 'nome_limpo', 'nome_temp', 'nome_norm', 'chave', # Strings auxiliares
    'mortes_cardio', # Vazamento: compoe a taxa
    'taxa_mortalidade', # Vazamento: origem do target
    'leitos_sus', 'total_medicos', 'estabelecimentos', # Redundante: ja existem as taxas por 100k
    'pib_total' # Redundante: ja existe o per capita
]

# Remove colunas irrelevantes
df_model = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

print(f'Dimensoes para modelagem: {df_model.shape}')
df_model.head()

Dataset carregado com sucesso. Dimensoes: (16783, 28)

Distribuicao da Classe Alvo:
target
1    0.629625
0    0.370375
Name: proportion, dtype: float64

Registros pertencentes as 100 maiores cidades: 324
Dimensoes para modelagem: (16783, 20)


,ano,populacao,tx_envelhecimento,codmun,idhm,idhm_renda,idhm_longevidade,idhm_educ,renda_pc,tx_analfabetismo,leitos_por_mil,medicos_por_100k,estab_por_100k,pib_per_capita_calc,tem_hospital,cod_uf,cod_regiao,regiao,target,is_top_100
0,2007,26533.0,4.595,110001,0.483,0.637,0.7305,0.3940,424.07,15.21,1.696001,678.400482,45.226699,7212.301662,1,11,1,Norte,0,0
1,2010,24392.0,5.840,110001,0.641,0.657,0.7630,0.5260,476.99,13.00,1.844867,828.140374,61.495572,10744.383404,1,11,1,Norte,0,0
2,2015,24392.0,5.840,110001,0.641,0.657,0.7630,0.5260,476.99,13.00,1.844867,1078.222368,98.392916,17272.056412,1,11,1,Norte,0,0
3,2007,74503.0,3.640,110002,0.556,0.695,0.7740,0.4715,610.41,10.67,0.885870,744.936446,59.058025,12149.886582,1,11,1,Norte,0,0
4,2010,90353.0,4.360,110002,0.702,0.716,0.8060,0.6000,689.95,8.53,0.730468,913.085343,92.968689,15104.025323,1,11,1,Norte,0,0


## Divisão Temporal 
- Divisao Temporal (Out-of-Time Validation)
- Treino: Dados historicos (2007 e 2010)
- Teste: Dados futuros (2015) para simular previsao real


In [10]:
train = df_model[df_model['ano'] < 2015].copy()
test = df_model[df_model['ano'] == 2015].copy()

print(f'Conjunto de Treino (Anos 2007, 2010): {train.shape}')
print(f'Conjunto de Teste (Ano 2015): {test.shape}')

# Separacao de X (Features) e y (Target)
# Remove identificadores das features para evitar memorizacao pelo modelo
cols_identificadores = ['target', 'codmun', 'ano', 'cod_uf', 'cod_regiao'] 

X_train = train.drop(columns=cols_identificadores)
y_train = train['target']

X_test = test.drop(columns=cols_identificadores)
y_test = test['target']

# Armazenamento de metadados para analise de erro posterior
meta_train = train[['codmun', 'ano', 'is_top_100', 'regiao']]
meta_test = test[['codmun', 'ano', 'is_top_100', 'regiao']]

print('\n[INFO] Features selecionadas para treinamento:')
print(list(X_train.columns))

Conjunto de Treino (Anos 2007, 2010): (11506, 20)
Conjunto de Teste (Ano 2015): (5277, 20)

[INFO] Features selecionadas para treinamento:
['populacao', 'tx_envelhecimento', 'idhm', 'idhm_renda', 'idhm_longevidade', 'idhm_educ', 'renda_pc', 'tx_analfabetismo', 'leitos_por_mil', 'medicos_por_100k', 'estab_por_100k', 'pib_per_capita_calc', 'tem_hospital', 'regiao', 'is_top_100']


## Tratamento de Categóricas 

Transformando "sul" e "norte" em numeros para o modelo entender 

In [11]:
# Identificacao de tipos de variaveis
cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()
num_cols = X_train.select_dtypes(exclude=['object']).columns.tolist()

print(f'Variaveis Categoricas detectadas: {cat_cols}')
print(f'Quantidade de Variaveis Numericas: {len(num_cols)}')

# Aplicacao de One-Hot Encoding para variaveis categoricas (ex: Regiao)
# drop_first=True evita multicolinearidade para modelos lineares
X_train_encoded = pd.get_dummies(X_train, columns=cat_cols, drop_first=True)
X_test_encoded = pd.get_dummies(X_test, columns=cat_cols, drop_first=True)

# Alinhamento de colunas entre Treino e Teste
# Garante que o teste tenha a mesma estrutura do treino, preenchendo ausentes com 0
X_train_encoded, X_test_encoded = X_train_encoded.align(X_test_encoded, join='left', axis=1, fill_value=0)

print(f'Shape Final X_train: {X_train_encoded.shape}')
print(f'Shape Final X_test: {X_test_encoded.shape}')

Variaveis Categoricas detectadas: ['regiao']
Quantidade de Variaveis Numericas: 14
Shape Final X_train: (11506, 18)
Shape Final X_test: (5277, 18)


## Salvar Arquivos 

Agora os arquivos estao processados em formato Parquet e CSV para a etapa de modelagem

In [12]:
try:
    # Features (Parquet para performance)
    X_train_encoded.to_parquet(MODEL_DATA_PATH / 'X_train.parquet')
    X_test_encoded.to_parquet(MODEL_DATA_PATH / 'X_test.parquet')

    # Targets (CSV)
    y_train.to_frame('target').to_csv(MODEL_DATA_PATH / 'y_train.csv', index=False)
    y_test.to_frame('target').to_csv(MODEL_DATA_PATH / 'y_test.csv', index=False)

    # Metadados (CSV)
    meta_train.to_csv(MODEL_DATA_PATH / 'meta_train.csv', index=False)
    meta_test.to_csv(MODEL_DATA_PATH / 'meta_test.csv', index=False)

    print('[SUCESSO] Dados preparados e salvos no diretorio data/model_input.')
    print('[INFO] Pipeline de preparacao concluido. Pronto para treinamento.')
    
except Exception as e:
    print(f'[ERRO] Falha ao salvar arquivos de modelagem: {e}')

[SUCESSO] Dados preparados e salvos no diretorio data/model_input.
[INFO] Pipeline de preparacao concluido. Pronto para treinamento.
